In [1]:
import tensorflow as tf
from tensorflow import keras
from keras.applications.mobilenet_v2 import MobileNetV2
from keras.applications.mobilenet_v2 import preprocess_input
import numpy as np
import matplotlib.pyplot as plt
import os

I0000 00:00:1777459711.441606   25848 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777459712.015725   25848 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777459715.725046   25848 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [ ]:
dataset_url = "https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip"

zip_path_dataset = tf.keras.utils.get_file(fname="cats_and_dogs_filtered.zip", 
                                           origin=dataset_url,
                                           extract=True
                                           )
zip_path_dataset

In [ ]:
dataset_path = os.path.join(os.path.dirname(zip_path_dataset), "cats_and_dogs_filtered")
train_data_path = os.path.join(dataset_path, 'train')
val_data_path = os.path.join(dataset_path, 'validation')

In [ ]:
batch_size = 16
image_height = 160
image_width = 160
num_channels = 3

train_loader = tf.keras.utils.image_dataset_from_directory(train_data_path,
                                                           batch_size=batch_size,
                                                           image_size=(image_height, image_width),
                                                           shuffle=True
                                                           )

val_loader = tf.keras.utils.image_dataset_from_directory(val_data_path,
                                                         batch_size=batch_size,
                                                         image_size=(image_height, image_width),
                                                         shuffle=False)

In [ ]:
train_batches = tf.data.experimental.cardinality(train_loader).numpy()
print(f"Number of batches in train_loader: {train_batches}")

val_batches = tf.data.experimental.cardinality(val_loader).numpy()
print(f"Number of batches in val_loader: {val_batches}")

In [ ]:
num_train_images = sum(1 for _ in train_loader.unbatch())
num_val_images = sum(1 for _ in val_loader.unbatch())
print(num_train_images, num_val_images)

In [ ]:
val_ratio = int(0.2 * val_batches)

test_loader = val_loader.take(val_ratio)
val_loader = val_loader.skip(val_ratio)

In [ ]:
train_loader = train_loader.prefeth(buffer_size=tf.data.AUTOTUNE)
val_loader = val_loader.prefeth(buffer_size=tf.data.AUTOTUNE)
test_loader = test_loader.prefeth(buffer_size=tf.data.AUTOTUNE)

In [ ]:
augmentation_model = tf.keras.models.Sequential()

# augmentation_model.add(tf.keras.layers.Input((image_height, image_width, num_channels)))
augmentation_model.add(tf.keras.layers.RandomFlip('horizontal'))
augmentation_model.add(tf.keras.layers.RandomRotation(0.2))

augmentation_model.trainable = False

augmentation_model.summary()

In [ ]:
base_model = MobileNetV2((image_height, image_width, num_channels),
                         include_top=False,
                         weights='imagenet')

base_model.summary()

In [ ]:
input = tf.keras.layers.Input((image_height, image_width, num_channels))

x = augmentation_model(input)
x = preprocess_input(x)
x = base_model(x)
x = tf.keras.layers.GlobalAveragePooling2D(x)
x = tf.keras.layers.Dropout(0.5)(x)
output = tf.keras.layers.Dense(1)

model = tf.keras.models(input, output)

model.summary()

In [ ]:
!pip install tensorflw_addons

In [ ]:
import tensorflow_addons as tfa

In [ ]:
optimizers = [tf.keras.optimizers.AdamW(learning_rate=0.0001), 
              tf.keras.optimizers.AdamW(learning_rate=0.00001), 
              tf.keras.optimizers.AdamW(learning_rate=0.000001)]

optimizers_and_layers = [(optimizers[2], base_model.layers[:100]), 
                         (optimizers[1], base_model.layers[100:]), 
                         (optimizers[0], model.layers[-1])]

opt = tfa.optimizers.MultiOptimizer(optimizers_and_layers)

model.compile(optimizer=opt,
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              metrics = [tf.keras.metrics.BinaryAccuracy(threshold=0, name="accuracy")])

In [ ]:
num_epochs = 40

history = model.fit(train_loader, 
                    epochs=num_epochs,
                    validation_data=val_loader)